# GNSS Timeseries Project

## Documentation overview
This notebook reads and displays velocity fields and time series from the NGF (EarthScope data center), UNR (University Nevada Reno), and JPL (Jet Propulsion Laboratory).  Most likely additional packages such as folium, pandas, numpy, ipywidgets, matplotlib, tabulate, geopy, json will need to be installed.  Conda installations from conda-forge should be possible for all packages.

The widget interface at the bottom of the notebook has three cells:

<b>Availability:</b> The “Latest NGF”, “Latest UNR”, and “Latest JPL” buttons must be used the first time the Notebook is run to create the data directory and subdirectories for each center that will used to save JSON files with information about all sites at the centers and time series files from each center as they are requested.  Latest updates should be selected to ensure the most up-to-date information is available.  The Availability check can used to see what is known about a 4-char code site name at each of the centers.

<b>Map:</b> The Map interface allows locations of sites, along with other sites within a user-specified radius to be plotted, along with 3-D velocities if available and selected.  Distance table for the 10 nearest sites to a site can be generated.

<b>Timeseries:</b> The Time series interface allows time series to be detrended, with optionally breaks from the EarthScope database removed.  Time series from different sites and centers can be overlayed.  Different backends can be used for plotting time series, with the ipympl backend allowing zooming and saving and the interactive one allowing identification of points.  The default inline backend should work on all systems.  The other backends may not work on all systems. 

All of the cells in the Notebook can be run, and the widgets and additional documentation will be at the bottom of the Notebook.  The code should run in a Jupyter Notebook or VSCode.



## Internal Code


### Importing Packages

In [ ]:
import folium as fol
from folium.features import DivIcon
import pandas as pd
import numpy as np
import ipywidgets as wg
from ipywidgets import HBox, VBox, HTML, Layout
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import gridspec
from tabulate import tabulate
import math
import statistics
import geopy.distance
from geopy.distance import geodesic
import json
import os
from os.path import exists
from os import makedirs
import sys
from datetime import datetime, timedelta, timezone
import random
import urllib.request
import requests
import textwrap
from pathlib import Path
# See which version of earthscope_sdk is available
try:
    import earthscope_sdk
    print('earthscope_sdk aleady installed in virtual environment')
except:
    # Install earthscope_sdk
    print('Installing earthscope_sdk in virtual environment')
    %pip install earthscope_sdk

import earthscope_sdk
#from earthscope_sdk.auth.device_code_flow import DeviceCodeFlowSimple
#from earthscope_sdk.auth.auth_flow import NoTokensError
# Now get the Earthscope_sdk version
esver = earthscope_sdk.__version__
print('Earthscope_sdk version',esver)
if( esver[0] == '1' ) :
    print('Newer version')


### Setting up Directories

In [ ]:
orglist = ["NGF","UNR", "JPL"]

def make_if_absent(folderpath):
    if not exists(folderpath): makedirs(folderpath)

make_if_absent("data")
for orgname in orglist:
    make_if_absent("data/"+orgname+"/")

### Reading from Existing Data

In [ ]:
data_of = {}

for org in orglist:
    filepath = "data/" + org + "/" + org + "_data.json"
    if exists(filepath):
        with open(filepath, "r") as data_file:
            data_of[org] = json.load(data_file)

### General Functions

In [ ]:
def FitTS(xdata,ydata,sig):
    '''Fit times series.  Linear trend currently
    data is xdata time in days, ydata (NEU), sig
    https://en.wikipedia.org/wiki/Generalized_least_squares
    https://en.wikipedia.org/wiki/Reduced_chi-squared_statistic
    Outputs:
    array([grad, y-int]), [grad, grad_sd], [Wrms, chi, ndata]
    '''

    # yr = xdata/365.25; # Years Since 2000/1/1
    # OmInv = 1./sig**2
    # ndata = int(xdata.size)  # number of data points
    # X = np.vstack((np.ones(ndata),yr)).T
    # Bvec = (X.T * OmInv) @ ydata.T
    # MCov = np.linalg.inv((X.T * OmInv) @ X)
    # MEst = MCov @ Bvec
    # Res = ydata - X @ MEst

    # chi = np.sqrt((Res.T @ (Res*OmInv))/(ndata-2))
    # wrms = np.sqrt(ndata/np.sum(OmInv))*chi

    # return np.flip(MEst), [MEst[1],np.sqrt(MCov[1,1])], [wrms,chi,ndata]

    # Set up to fit linear trend.  In later updates we could
    # more elaborate models with offsets, periodic and post-
    # seismic componensts.
    yr = xdata/365.25; # Years Since 2000/1/1
    wgh = 1./sig**2
    ndata = int(xdata.size)  # number of data points
    A = np.transpose([np.ones(ndata),yr])
    NormEQ = np.matmul(np.transpose(A)*wgh,A)
    Bvec = np.matmul(np.transpose(A)*wgh,np.transpose(ydata))
    MCov = np.linalg.inv(NormEQ)
    MEst = np.matmul(MCov,Bvec)
    Res = ydata-np.matmul(A,MEst)
    chi = np.sqrt(np.dot(np.transpose(Res),Res*wgh)/(ndata-2))
    wrms = np.sqrt(ndata/np.sum(wgh))*chi
    pfe = np.flip(MEst) # Set reverse order like polyfit
    #print("Est  : a=%5.3f, b=%5.3f" % tuple(MEst) )
    #print("Sig  : a=%5.3f, b=%5.3f" % tuple(np.sqrt(np.diag(MCov))) )
    #print('Fits',np.flip(MEst), [MEst[1],np.sqrt(MCov[1,1])], [wrms,chi,ndata])
    #print('xdata',xdata[0],xdata[-1],'ydata',ydata[0],ydata[-1])

    return pfe, [MEst[1],np.sqrt(MCov[1,1])], [wrms,chi,ndata]

def detrended(xdata, ydata, sig):
    # m, c = FitTS(xdata, ydata, sig)[0]
    # return ydata - (m * xdata / 365.25 + c)
    pfe , Vel, Stat = FitTS(xdata, ydata, sig)
    # Return the residuals after removing linear trend and the 
    # Velocity estimate with sigma and the Statistics (WRMS, NRMS, #data)
    return ydata - (pfe[0] * xdata / 365.25 + pfe[1]), Vel, Stat


def remove_outliers_function(nd, ed, ud, ns, es, us, times, td, multiplier):
    bool_list = [all(abs(lst[i]-np.mean(lst)) <= multiplier*np.std(lst) for lst in (nd, ed, ud)) for i in range(len(nd))]
    nd_new = nd[bool_list]
    ed_new = ed[bool_list]
    ud_new = ud[bool_list]
    ns_new = ns[bool_list]
    es_new = es[bool_list]
    us_new = us[bool_list]
    times_new = times[bool_list]
    td_new = td[bool_list]
    return nd_new, ed_new, ud_new, ns_new, es_new, us_new, times_new, td_new

# Code depending on which version of earthscpoe_sdk
if ( esver[0] == '0' ) :
    import requests
    from pathlib import Path
 
    from earthscope_sdk.auth.device_code_flow import DeviceCodeFlowSimple
    from earthscope_sdk.auth.auth_flow import NoTokensError

    def get_es_file(url, directory_to_save_file='./data/NGF/', token_path='./'):
        """function to get earthscope data using es-sdk

        Parameters
        ----------
        url : string
            url of desired file at gage-data.earthscope.org
        directory_to_save_file : str, optional
            path of directory in which to save the file, by default cwd
        token_path : str, optional
            path of directory in which to save the token, by default './'
        """
        # instantiate the device code flow subclass
        device_flow = DeviceCodeFlowSimple(Path(token_path))
        try:
            # get access token from local path
            device_flow.get_access_token_refresh_if_necessary()
        except NoTokensError:
            # if no token was found locally, do the device code flow
            device_flow.do_flow()
        
        token = device_flow.access_token

        # request a file and provide the token in the Authorization header
        file_name = Path(url).name

        r = requests.get(url, headers={"authorization": f"Bearer {token}"})
        if r.status_code == requests.codes.ok:
            # save the file
            with open(Path(Path(directory_to_save_file) / file_name), 'wb') as f:
                for data in r:
                    f.write(data)
        else:
            #problem occured
            print(f"failure: {r.status_code}, {r.reason}")
        
elif ( esver[0] == '1') :
    from pathlib import Path
    import time
    import requests
    from earthscope_sdk import EarthScopeClient
    
    BASE_URL = "https://data.earthscope.org/archive/gnss/products/position"

    def build_csv_url(station: str, author: str = "cwu", frame: str = "igs14") -> str:
        """
        Build the URL for a non-detrended GNSS position time-series CSV.

        Example:
            https://data.earthscope.org/archive/gnss/products/position/P162/P162.cwu.igs14.csv
        """
        st = station.strip().upper()
        return f"{BASE_URL}/{st}/{st}.{author.lower()}.{frame.lower()}.csv"

    def fetch_csv(url: str, outpath: Path, headers=None, retries: int = 3, backoff: float = 1.5):
        """Download a CSV with retries and basic validation."""
        err = ""
        for attempt in range(1, retries + 1):
            try:
                r = requests.get(url, headers=headers, timeout=30)
                if r.ok:
                    outpath.parent.mkdir(parents=True, exist_ok=True)
                    outpath.write_bytes(r.content)
                    return (url, True, "")
                err = f"HTTP {r.status_code} {r.reason}"
            except requests.RequestException as e:
                err = str(e)
            time.sleep(backoff ** attempt)
        return (url, False, err)

    def download_es_file(
        es_client: EarthScopeClient,
        station,
        frame="igs14",
        outdir="./data/NGF"):
        """
        Download non-detrended GNSS position time-series CSVs from EarthScope archive.
        Works with earthscope-sdk >=1.0.0
        Requires that you have run `es login` or configured credentials beforehand.
        """

        outdir = Path(outdir)
        outdir.mkdir(parents=True, exist_ok=True)

        # Parse author/station combos

        token = es_client.ctx.auth_flow.access_token
        headers = {"Authorization": f"Bearer {token}"}

        failures = []
        url = build_csv_url(station, "cwu", frame)
        fname = Path(url).name
        outpath = outdir / fname
        url, ok, err = fetch_csv(url, outpath, headers=headers)
        if ok:
            print(f"✓ {fname}")
        else:
            print(f"✗ {fname}  --  {err}")
            failures.append((fname, err))

        if failures:
            print("\n⚠️ Some downloads failed — check author/station/frame availability.")
        #else:
            #print("\n✅ All downloads completed successfully.")

    print('New EarthScope_SDK Version ',esver,' Set up')

else:
    print('Unknown earthscope_sdk version ',esver)
    
def calc_distance_earth(start_lat, start_lon, distance_km, direction = "east"):

    r = 6371.0 # radius of Earth
    # Convert latitude from degrees to radians
    if direction == "east":
        lat_rad = math.radians(start_lat)
        # Distance in kilometers corresponds to the change in longitude at the equator
        delta = (distance_km/(r * math.cos(lat_rad))) * (180/math.pi)
        # Calculate the new longitude
        end_lon = start_lon + delta
        # Latitude remains the same because we are moving directly east
        end_lat = start_lat

    elif direction == "west":
        lat_rad = math.radians(start_lat)
        # Distance in kilometers corresponds to the change in longitude at the equator
        delta = (distance_km/(r * math.cos(lat_rad))) * (180/math.pi)
        # Calculate the new longitude
        end_lon = start_lon - delta
        # Latitude remains the same because we are moving directly east
        end_lat = start_lat

    elif direction == "north":
        delta = distance_km / r
        # Convert latitude from degrees to radians
        lat_rad = math.radians(start_lat)
        # Calculate new latitude in radians
        new_lat_rad = lat_rad + delta
        # Convert new latitude from radians to degrees
        end_lat = math.degrees(new_lat_rad)
        # Longitude remains the same
        end_lon = start_lon

    elif direction == "south":
        delta = distance_km / r
        # Convert latitude from degrees to radians
        lat_rad = math.radians(start_lat)
        # Calculate new latitude in radians
        new_lat_rad = lat_rad - delta
        # Convert new latitude from radians to degrees
        end_lat = math.degrees(new_lat_rad)
        # Longitude remains the same
        end_lon = start_lon

    return [end_lat, end_lon]


### Timeseries Readers

In [ ]:
## FETCH TIMESERIES DATA 

SigLim = (10,10,30) # standard deviation

# UNR Data Fetcher
def Read_UNR(site, Update):
    # !! check for if site has no network
    site_info = data_of["UNR"][site]
    # http://geodesy.unr.edu/gps_timeseries/tenv3/IGS14/VMAG.tenv3
    # MOD TAH 241223: Use IGS14 time series.
    #file_directory = "./data/UNR/"+site.upper()+"."+site_info["regions"][0]+".csv"
    #file_name = site.upper()+"."+site_info["regions"][0]+".tenv3"
    #fetch_url="http://geodesy.unr.edu/gps_timeseries/tenv3/plates/"+site_info["regions"][0]+"/"+file_name
    file_directory = "./data/UNR/"+site.upper()+".csv"
    file_name = site.upper()+".tenv3"
    #fetch_url="https://geodesy.unr.edu/gps_timeseries/tenv3/IGS14/"+file_name
    fetch_url="https://geodesy.unr.edu/gps_timeseries/IGS20/tenv3/IGS20/"+file_name

    if exists(file_directory) and not bool(Update) :
        print('Reading from file ',file_directory)
        df = pd.read_csv(file_directory,delimiter=',')
    else:
        print('Downloading from ',fetch_url)
        try:
            df = pd.read_csv(fetch_url,delimiter=r"\s+",header=1)
            df.to_csv(file_directory,index=False)
        except:
            df = []
            with timeseries_output:
                display(wg.HTML('''<em style="color:red">UNR Site '''+site.upper()+''' cant be found</em>'''))

    ninit = len(df)
    # Remove data with NEU sigmas > 10, 10 and 30 mm.
    # Apply sequencially because 'and' does not seem to work
    if SigLim[0] > 0:
        df=df[df.iloc[:,14]<SigLim[0]/1000]
        df=df[df.iloc[:,15]<SigLim[1]/1000]
        df=df[df.iloc[:,16]<SigLim[2]/1000]
        print("Number after >{} {} {} mm NEU sigma removal".format(*SigLim),len(df),"Read",ninit)
    npdat = df.to_numpy()

    # Extract time
    t=list(npdat[:,1])
    nd=list((npdat[:,10]-npdat[0,10])*1000) # remove first value so starts at zero
    # the same a NGF. (Problem if times of first data point are different)
    ed=list((npdat[:,8]-npdat[0,8])*1000)
    ud=list((npdat[:,12]-npdat[0,12])*1000)
    ns=list(npdat[:,15]*1000) ; es=list(npdat[:,14]*1000) ; us=list(npdat[:,16]*1000)

    n = 0
    to = []
    td = np.zeros(len(t))
    for v in t:
        to = np.append(to, datetime.strptime(v, '%y%b%d')+timedelta(hours=12))
        dt = to[n] - datetime(2000, 1, 1,0,0)  # Time difference from 2000/1/1
        td[n] = dt.total_seconds()/86400.  # Days from 2000/1/1
        n += 1

    tseries = np.array([td,nd,ns,ed,es,ud,us])
    times = to
    return times, tseries

def Read_JPL(site, Update):
    # !! check for if site has no network
    csv_file_directory = "./data/JPL/"+site+".csv"

    fetch_url="https://sideshow.jpl.nasa.gov/pub/JPL_GPS_Timeseries/repro2018a/post/point/"+site+".series"

    if exists(csv_file_directory) and not bool(Update):
        print('Reading from file',csv_file_directory)
        df = pd.read_csv(csv_file_directory,delimiter=',')
    else:
        print('Downloading from ',fetch_url)
        try:
            df = pd.read_csv(fetch_url,delimiter=r"\s+")
            df.to_csv(csv_file_directory,index=False)
        except:
            df=[]
            with timeseries_output:
                display(wg.HTML('''<em style="color:red">JPL Site '''+site.upper()+''' cant be found</em>'''))

    ninit = len(df)
    # Remove data with NEU sigmas > 10, 10 and 30 mm.
    # Apply sequencially because 'and' does not seem to work
    if SigLim[0] > 0:
        df=df[df.iloc[:,5]<SigLim[0]/1000]
        df=df[df.iloc[:,4]<SigLim[1]/1000]
        df=df[df.iloc[:,6]<SigLim[2]/1000]
        print("Number after >{} {} {} mm NEU sigma removal".format(*SigLim),len(df),"Read",ninit)

    npdat = df.to_numpy()

    nd=list((npdat[:,2]-npdat[0,2])*1000) # 3rd column, remove first value so starts at zero
    # the same a NGF. (Problem if times of first data point are different)
    ed=list((npdat[:,1]-npdat[0,1])*1000) # 2nd column
    ud=list((npdat[:,3]-npdat[0,3])*1000) # 4th column
    ns=list(npdat[:,5]*1000) ; es=list(npdat[:,4]*1000) ; us=list(npdat[:,6]*1000) # what do any of these mean
    to = np.empty(len(npdat[:,10]), dtype = object)

    for i in range(len(to)):
        each_date = datetime((int(npdat[:,11][i])), int((npdat[:,12][i])), int((npdat[:,13][i])))+timedelta(hours=12)
        to[i] = each_date

    td_counter = 0
    td = np.zeros(len(npdat[:,10]))
    for time_value in npdat[:,10]:
        td[td_counter] = time_value/86400 # days from 2000/1/1 (file has seconds from 2000/1/1 12:00 hr)
        td_counter += 1

    tseries = np.array([td,nd,ns,ed,es,ud,us])
    times = to
    return times, tseries




def Read_NGF(site, Update):
    '''Function to read data from Earthscope NGF and return numpy array and datetime arrays.
    Input:
       sites - 4-character site name
       Update - 1 Re-read data from server; 0 Use existing copy
    Output
       times   - date-times dates thar can be used in plotting
       tseries - time series with days for 2000/1/1, dn,sn, de,se, du, su'''
    global SigLim
    # Incase of token refresh problems may need
    #!es sso refresh

    # Create the file name and URL
    # filen = site.upper()+".cwu.igs14.csv"
    filen = site.upper()+".cwu.igs14.csv"

    filepath = "data/NGF/" + filen
    whurl="https://data.unavco.org/archive/gnss/products/position/"+site.upper()+"/"+filen

    #PBO Station Position Time Series.
    #Format Version, 1.2.0
    #Reference Frame, igs14
    #4-character ID, P177
    #Station name, CoDeTierraCN2008
    #Begin Date, 2008-05-09
    #End Date, 2021-10-06
    #Release Date, 2021-10-07
    #Source file, P177.cwu.igs14.pos
    #offset from source file, 166.01 mm North, -139.45 mm East, -3.62 mm Vertical
    #Reference position, 37.5281683684 North Latitude, -122.4950535711 East Longitude, 71.79172 meters elevation
    #Date, North (mm), East (mm), Vertical (mm), North Std. Deviation (mm), East Std. Deviation (mm), Vertical Std. Deviation (mm), Quality,
    #2008-05-09,0.00, 0.00, 0.00, 2.08, 1.66, 7.84, repro,
    #2008-05-10,0.38, 0.83, 2.66, 2.13, 1.7, 7.99, repro,
    #...
    # print(filen, whurl)
    if exists(filepath) and not bool(Update):
        print('Reading from file',filen)
        df = pd.read_csv(filepath,delimiter=',',header=11)
    else:
        if ( esver[0] == '0' ) :
            print('Downloading from ',whurl)
            try:
                get_es_file(whurl)
                df = pd.read_csv(filepath,delimiter=',',header=11)
            except:
                df = []
                with timeseries_output:
                    display(wg.HTML('''<em style="color:red">NGF Site '''+site.upper()+''' cant be found</em>'''))

            # Write csv local copy for later use
            #print('Creating',filen)
            #df.to_cvs(filen)
            # df.to_csv(filen,index=False)
        else:
            # Use ES 1.0 verions
            print('Getting',site.upper(),'with EarthSopeClient')
            es = EarthScopeClient()
            # Refresh access token if necessary -
            #
            # If this fails, you'll need to use the EarthScope CLI again to login: es login
            # Example error: "NoRefreshTokenError: No refresh token was found. Please re-authenticate."
            es.ctx.auth_flow.refresh_if_necessary()
            # Download time series for given author/station pairs
            download_es_file(es, site.upper(), frame="igs14")

        df = pd.read_csv(filepath,delimiter=',',header=11)
        # Write csv local copy for later use
        #print('Creating',filen)
        #df.to_cvs(filen)
        # df.to_csv(filen,index=False)

    ninit = len(df)
    # Remove data with NEU sigmas > 10, 10 and 30 mm.
    # Apply sequencially because 'and' does not seem to work (latter use values
    # extracted from GUI box)
    if SigLim[0] > 0 :
        df=df[df.iloc[:,4]<SigLim[0]];
        df=df[df.iloc[:,5]<SigLim[1]];
        df=df[df.iloc[:,6]<SigLim[2]];
        print("Number after >{} {} {} mm NEU sigma removal".format(*SigLim),len(df),"Read",ninit)

    npdat =df.to_numpy()
    # Extract time,and data and sigma
    t=list(npdat[:,0]);nd=list(npdat[:,1]); ed=list(npdat[:,2]); ud=list(npdat[:,3]);
    ns=list(npdat[:,4]) ; es=list(npdat[:,5]) ; us=list(npdat[:,6])
    # Now convert the time into a datetime type (to) and days from 2000/01/01
    # as float array (td).
    # We can use 'to' in plotting and 'td' for fitting to data
    n = 0
    to = []
    td = np.zeros(len(t))
    for v in t:
        # Move time to middle of the day.
        to = np.append(to,datetime.strptime(v, '%Y-%m-%d')+timedelta(hours=12))
        dt = to[n] - datetime(2000, 1, 1)  # Time difference from 2000/1/1
        td[n] = dt.total_seconds()/86400.  # Days from 2000/1/1
        n += 1

    # Now construct data array
    tseries =  np.array([td,nd,ns,ed,es,ud,us])
    times = to
    return times, tseries

Read = {
        "NGF": Read_NGF,
        "UNR": Read_UNR,
        "JPL": Read_JPL
    }

### Map Functions

In [ ]:
## GENERAL MAP OPERATIONS

d_km = lambda coords1, coords2: geopy.distance.geodesic(coords1,coords2).km
d_mi = lambda coords1, coords2: geopy.distance.geodesic(coords1,coords2).miles

def plot_locations(map, coordslist, tooltiplist, color="blue", dotrad = 3):
    """
    map: Folium map
    coordslist: [(lat1, lon1), (lat2,lon2), ...]
    tooltip: [tag1, tag2, ...]
    """
    for coords,tooltip in zip(coordslist,tooltiplist):
        # print(coords, tooltip)
        fol.CircleMarker(
                location=coords,
                tooltip=tooltip,
                radius = dotrad,
                stroke = False,
                fill_color = color,
                fill_opacity = 1,
                opacity = 1,
                popup=tooltip
                #icon=fol.Icon(icon="globe"),
            ).add_to(map)


def nearby_sites(org, site, radius_in_km):
    """
    site = ID of a site OR coordinates provided as a list
    radius_in_km = radius within which we want to plot sites
    Returns: [ (neighbor_property, lat, lon, distance from site), ... ]
    """
    if site[0] in "([": # if it's a coordinate pair instead of a site id
        site = list(eval(site))

    cur_coords = site

    if isinstance(site,str):
        cur_coords = data_of[org][site]["location"]

    assert not isinstance(cur_coords[0],str) # by now cur_coords is a coordinate pair regardless of input

    id_coord_dist_list = []

    for siteid in data_of[org]:
        try:
            site_coords = data_of[org][siteid]["location"]
            distance = d_km(site_coords, cur_coords)
            if distance < radius_in_km : #  and site != siteid:  (siteid site as well for tooltip)
                id_coord_dist_list.append((siteid,*site_coords,distance))
        except: pass

    return id_coord_dist_list, cur_coords


def basic_circle(map, center, radius_in_km):
    fol.Circle(
        location=center,
        radius=radius_in_km * 1000, # in metres
        color="black",
        weight=1,
        fill_opacity=0.0,
        opacity=1,
        fill_color="green",
        fill=True,  # gets overridden by fill_color
        #tooltip="I am in meters",
    ).add_to(map)

# start with point, velocity
# - default is that the length of the vector should be 1 mm = 1 km?
# -

def vec_add(c1,c2):
    return [c1[i]+c2[i] for i in range(len(c1))]

def vec_sub(c1,c2):
    # Compute c1-c2
    return [c1[i]-c2[i] for i in range(len(c1))]

def draw_vector(map, center, direction, scale, color="black",label="None"):

    # direction = velocity = [n,e,u]

    # vec_length = math.hypot(direction[0], direction[1])

    # if scale = 1, 1 mm = 1 km
    dlat = calc_distance_earth(center[0], center[1], abs(direction[0]*scale), "south" if direction[0] < 0 else "north")[0] # north/south
    dlon = calc_distance_earth(center[0], center[1], abs(direction[1]*scale), "west" if direction[1] < 0 else "east")[1] # east/west

    endpoint = [dlat, dlon]

    # dvec = [direction[0] * scale * abs(math.cos(center[0] * math.pi/180 )), direction[1] * scale]
    # #velocities are in mm per year

    # endpoint = vec_add(center, dvec)

    fol.PolyLine(locations=[center, endpoint],weight=2,color = color).add_to(map)
    
    if( label != "None") :
        fol.map.Marker(endpoint,
            icon=DivIcon(
                icon_size=(250,36),
                icon_anchor=(0,0),
                html='<div style="font-size: 12pt">'+label+'</div>',
            )
        ).add_to(map)


# add functionality to let ppl add circles too, and specify location of them


### Timeseries Functions

In [ ]:
# Functions for editing and plotting time series.
def remove_brac(string):
    '''
    "P123 (UNR)" --> ("P123", "UNR")
    '''
    i=-1
    while string[i] != '(': i-=1
    return string[:i-1], string[i+1:-1]


def plot_ts_graph(siteid, org, Update, detrend, ax0, ax1, ax2, yearrange, breaks, errorbars, errorbar_outline, outlier, shift, color):
    '''
    Adding a single plot to the original plot defined in plot_ts_graph_list()
    '''

    try:
        times, tseries = Read[org](siteid, Update)
    except:
        print('Error getting',siteid,'from',org)
        times = []; tseries = []
        return
 
    td, nd, ns, ed, es, ud, us = tseries

    ## restricting timeseries to yearrange
    # find the start and end index
    start, end = 0, len(td)
    while times[start] < yearrange[0]: start+=1
    while times[end-1] > yearrange[1]: end-=1
    # restrict

    # for name in [times, td, nd, ns, ed, es, ud, us]:
    #     name = name[start:end]
    times, td = times[start:end], td[start:end]
    nd, ns = nd[start:end], ns[start:end]
    ed, es = ed[start:end], es[start:end]
    ud, us = ud[start:end], us[start:end]
    
    if breaks:
        if "breaks" in data_of[org][siteid]:
            breaktimes_dict = {}
            for dt in data_of[org][siteid]["breaks"]:
                try:
                    yr_mm_day = [int(date) for date in dt[1:-1].split(",")][0:5]
                    breaktime = datetime(*eval(str(yr_mm_day)))
                    breaktimes_dict[breaktime] = [data_of[org][siteid]["breaks"][dt]["offsets"], int(np.where(times >= breaktime)[0][0])]
                except:
                    pass
            # [45.01, 0.67, 39.72, 0.59, -27.67, 1.83], [dN (mm), sN (mm), dE (mm), sE (mm), dU (mm), sU (mm)]
            interval_counter = 0
            breaktimes_list = list(breaktimes_dict)
            ndat = len(ud)
            while interval_counter in range(len(breaktimes_dict)):
                dict_key = breaktimes_list[interval_counter]
                interval_start = breaktimes_dict[dict_key][1]
                # Using -1 as last point skipped last point
                nd[interval_start:ndat] = nd[interval_start:ndat] - breaktimes_dict[dict_key][0][0]
                ed[interval_start:ndat] = ed[interval_start:ndat] - breaktimes_dict[dict_key][0][2]
                ud[interval_start:ndat] = ud[interval_start:ndat] - breaktimes_dict[dict_key][0][4]
                interval_counter +=1
        else:
            print("No breaks information!")

    dfa = []
    if detrend:
        nd, VelN, StatN = detrended(td, nd, ns)
        ed, VelE, StatE = detrended(td, ed, es)
        ud, VelU, StatU = detrended(td, ud, us)

        if outlier: # while length of the array before and after are different, keep iterating the code
            initial_length = len(nd)
            length1 = 1
            length2 = 2
            iteration_cnt = 0
            while length1 != length2:
                # print(f"length of nd: {len(nd)}, {len(ud)}, {(len(ed))}, {len(td)}")
                # print(f"length of s: {len(ns)}, {len(us)}, {(len(es))}")
                length1 = len(nd)
                nd, ed, ud, ns, es, us, times, td = remove_outliers_function(nd, ed, ud, ns, es, us, times, td, outlier)
                # print(f"lengths after: {len(nd)}, {len(ud)}, {(len(ed))}, {len(td)}")
                # print(f"before: {np.max(ud)}, {np.min(ud)}")
                nd, dVelN, StatN = detrended(td, nd, ns)
                VelN[0] = VelN[0]+dVelN[0]   # Fit is to residuals, so update total

                ed, dVelE, StatE = detrended(td, ed, es)
                VelE[0] = VelE[0]+dVelE[0]   # Fit is to residuals, so update total
                ud, dVelU, StatU = detrended(td, ud, us)
                VelU[0] = VelU[0]+dVelU[0]   # Fit is to residuals, so update total

                length2 = len(nd)
                iteration_cnt +=1
                # print(f"after: {np.max(ud)}, {np.min(ud)}")
            print(f"Final {len(nd)} data; Number of outliers removed: {initial_length - len(nd)} with {iteration_cnt} iterations: ")

        # Now save the data frame information for the fit.
        VelNEU = VelN+VelE+VelU ; StatNEU = StatN[0:2]+StatE[0:2]+StatU
        labl = str(siteid)+'-'+str(org)
        # print('DFrame ',VelNEU,StatNEU,labl)
        dfa = pd.DataFrame([VelNEU+StatNEU], \
            columns=['Vn','σVn','Ve','σVe','Vu','σVu','WRMS N','χn','WRMS E','χe','WRMS U','χu','Num'],index=[labl])


    nd = nd+shift ;  ed = ed+shift ; ud = ud+shift*3

    if errorbars[0]:
        ax0.errorbar(times,nd,yerr=[ns,ns],errorevery=errorbars[1], capsize=2, color = color, linewidth=1) #, ecolor='black')
        ax1.errorbar(times,ed,yerr=[es,es],errorevery=errorbars[1], capsize=2, color = color, linewidth=1) #, ecolor='black')
        ax2.errorbar(times,ud,yerr=[us,us],errorevery=errorbars[1], capsize=2, color = color, linewidth=1) #, ecolor='black')

    if errorbar_outline[0]:
        for ax, d, s in [(ax0, nd, ns), (ax1, ed, es), (ax2, ud, us)]:
            ax.fill(list(times) + list(reversed(times)),
                list(d+s) + list(np.flip(d-s)),
                alpha=errorbar_outline[1], linewidth=1, color = color, label="_"+siteid)

    for ax, d in [(ax0, nd), (ax1, ed), (ax2, ud)]:
        ax.plot(times, d, linewidth=0.5, label=siteid + " (" + org + ")", color = color)
        if "breaks" in data_of[org][siteid]:
            for dt in data_of[org][siteid]["breaks"]:
                yr_mm_day = [int(date) for date in dt[1:-1].split(",")][0:3]
                breaktime = datetime(*eval(str(yr_mm_day)))
                try:
                    np.where(times >= breaktime)[0][0]
                    ax.axvline(x=breaktime, color='r', ls='--')
                except:
                    pass

    return dfa

def plot_ts_graph_list(idlist, Update, yearrange, breaks, detrend, errorbars, errorbar_outline, outlier, resolution = "Low Res", shift=0):
    '''
    Plots the timeseries of a list of sites.
    '''
    global tsfig
    tsfig = plt.figure(figsize=(15, 10), dpi = {"HD": 600, "Regular": 300, "Low Res": 100}[resolution])
    gs = gridspec.GridSpec(3, 1, height_ratios=[1, 1, 1])

    ax0 = plt.subplot(gs[0])
    ax0.set_ylabel("ΔNorth (mm)")
    ax0.yaxis.set_tick_params(labelrotation=90)
    plt.setp(ax0.get_xticklabels(), visible=False)

    ax1 = plt.subplot(gs[1], sharex = ax0)
    ax1.set_ylabel("ΔEast (mm)")
    ax1.yaxis.set_tick_params(labelrotation=90)
    plt.setp(ax1.get_xticklabels(), visible=False)

    ax2 = plt.subplot(gs[2], sharex = ax0)
    ax2.set_ylabel("ΔUp (mm)")
    ax2.yaxis.set_tick_params(labelrotation=90)
    ax2.set_xlabel("Time")

    colors = ["b", 'g', 'r', 'c', 'm', 'y', 'k', 'sienna']
    color_count = 0
    for idorg in idlist:
        dfl = plot_ts_graph(*remove_brac(idorg), Update, detrend, ax0, ax1, ax2, yearrange, breaks, errorbars, errorbar_outline, outlier, shift[idorg] if idorg in shift else 0, colors[color_count])
        if color_count < 8:
            color_count += 1
        else:
            color_count = 0

        # Create a pandas data frame will the results in it
        if detrend :
            if color_count == 1 :
                dfall = dfl
            else:
                dfall = pd.concat([dfall,dfl])
 
    # When all done display the data frame
    if detrend :
        # Format the DataFrame (Early versions of pandas may have problems)
        display(dfall.style.format({'Vn': '{:.2f}','σVn': '{:.3f}',
                                  'Ve': '{:.2f}','σVe': '{:.3f}',
                                  'Vu': '{:.2f}','σVu': '{:.3f}',
                                  'WRMS N': '{:.2f}','χn': '{:.2f}',
                                  'WRMS E': '{:.2f}','χe': '{:.2f}',
                                  'WRMS U': '{:.2f}','χu': '{:.2f}','Num':'{:d}'}))
 
        #display(dfall.style.format('{:.2f}')) 
        MD = False # Set true to get MarkDown formatted table.
        if MD :
            headers = ['Site','Vn','σVn','Ve','σVe','Vu','σVu','WRMS N','χn','WRMS E','χe','WRMS U','χu','Num']
            table = dfall.to_numpy()
            index = np.array(list(dfall.index.values))
            fulltable = np.c_[index , table]
            print('\nPIPE: For pandas.to_markdown')
            print(tabulate(fulltable, headers, tablefmt="pipe",floatfmt=("%s",".2f", ".3f",".2f",".3f",".2f", ".3f",".2f",".2f",".2f", ".2f",".2f",".2f",".2f")))


    # leg = ax0.legend(bbox_to_anchor=(1, 1))
    leg = ax0.legend()

    plt.subplots_adjust(hspace=.0)

    #for legobj in leg.legend_handles: # The lines in the legend were too thin previously.
    #    legobj.set_linewidth(3.0)

    plt.show()
    #return tsfig


### Widgets

#### Fetcher Widgets

In [ ]:
# Buttons
update_UNR_butt     = wg.Button(description="Latest UNR", icon = "download")
update_JPL_butt     = wg.Button(description="Latest JPL", icon = "download")
update_NGF_butt    = wg.Button(description="Latest NGF", icon = "download")
breaks_select   = wg.Dropdown(options=orglist, value='UNR', distabled = False, layout=wg.Layout(width='100px'))
add_breaks_site_button = wg.Button(description="Add Breaks")
remove_breaks_site_button = wg.Button(description="Remove Breaks")
# Outputs
fetcher_output      = wg.Output()
breaks_update_output = wg.Output()

#### Availability Widgets

In [ ]:
# Inputs
site_searchbar      = wg.Text(value=None,placeholder='Site ID',description='Check Availability:',disabled=False, style= {'description_width': 'initial'})
org_avail_select    = wg.Dropdown(options=orglist, value='NGF', disabled=False, layout=wg.Layout(width='100px'))
site_search_submit  = wg.Button(description="Search!", icon = "search")
clear_log_butt      = wg.Button(description="Clear Log", icon = "times")
# Style
site_search_submit.style.button_color = 'rgb(196,253,196)'
clear_log_butt.style.button_color = 'mistyrose'
# Outputs
availability_output = wg.Output()

#### Map Widgets

In [ ]:
# Inputs
map = ["Not initialized yet!"] # allow direct editing of the map via entry mutation
layout              = lambda w: wg.Layout(width=w, height='40px')
new_map_butt        = wg.Button(description="Clear / New Map", icon = "map")
reload_map_butt     = wg.Button(description="Reload Map", icon = "refresh")
close_map_butt      = wg.Button(description="Close Map", icon = "times")
site_id         = wg.Text(value=None,placeholder='Site ID',description='Site:',disabled=False, style= {'description_width': 'initial'}, layout=layout("150px"))
org_map_select      = wg.Dropdown(options=orglist+['other'], value='NGF', disabled=False, layout=wg.Layout(width='100px'))
site_radius         = wg.IntText(value=0,placeholder='in km',description='Radius:',disabled=False, style= {'description_width': 'initial'}, layout=layout("150px"))
plot_graph_butt     = wg.Button(description="Distance Table", icon = "line-chart")
vel_siglim_form     = wg.Text(value=None, description="Vel SigLim:", placeholder='(1,1,2)', layout=wg.Layout(width='200px'))
plot_vec_check      = wg.Checkbox(description="Plot Velocities", value=False, layout = layout("200px"))
plot_rel_vel        = wg.Checkbox(description="Relative Vel", value=False,layout = layout("200px")  )
arrow_length_input    = wg.FloatText(description = "Arrow mm/yr", layout = layout("150px"), disabled = False, value = 10,
                                     tooltip="Length of arrow shown for scale")
#length_coords_text  = wg.Text(description = "Starting Coords:", disabled = False, value = "0,0", layout=(layout("150px")))
#draw_length_button  = wg.Button(description = "Draw Length:")
velocity_scale_input   = wg.FloatText(description = "Vel Scale", layout=layout("150px"), value = 10,
                                      tooltip='Scale from mm/yr to km on map')
colorscalefactor_input = wg.FloatText(description = "±U Rate", layout=layout("150px"), value = 5,
                                      tooltip='±<Range for Vertical Rate> (color saturates after this value) red (negative) to blue (positive)')
thin_velocities_input  = wg.FloatText(description = "Decimate", layout=layout("150px"), value = 1)

# map radius lists
map_radius_list     = wg.SelectMultiple(options=[], value=[], description = "Site/Radius:", )
add_sites_map_button  = wg.Button(description = "Add Sites")
remove_sites_map_button = wg.Button(description = "Remove Site(s)")
clear_map_list_butt     = wg.Button(description="Clear All", icon = "times")

site_circle_submit  = wg.Button(description="Add Plot!", icon = "map-marker")

# Style
new_map_butt.style.button_color = 'rgb(196,253,196)'
reload_map_butt.style.button_color = 'oldlace'
close_map_butt.style.button_color = 'mistyrose'
clear_map_list_butt.style.button_color = 'mistyrose'
site_circle_submit.style.button_color = 'paleturquoise'
plot_graph_butt.style.button_color = 'paleturquoise'

# Basemap options
BaseMap_butt = wg.RadioButtons(
    options=['OpenTopo', 'OpenStreetMap', 'ArcGIS Image'],
    value='OpenTopo',
    description='Base Map Choice:',
    disabled=False
)
BaseMap_Set = wg.Button(description = "Set Base Map")


# Outputs
map_output          = wg.Output()
nearest_site_output = wg.Output()
graph_output        = wg.Output()

#### Timeseries Widgets

In [ ]:
# Inputs
ts_site_form        = wg.Text(value=None,placeholder='Site ID',disabled=False, style= {'description_width': 'initial'}, layout=layout("150px"))
org_ts_select       = wg.Dropdown(options=orglist+['other'], value='NGF', disabled=False, layout=wg.Layout(width='100px'))
append_butt         = wg.Button(description="Add to List", icon = "plus-square")
plot_ts_butt        = wg.Button(description="Plot", icon = "line-chart")
plot_ts_res         = wg.Dropdown(options=['HD', 'Regular', 'Low Res'], value='Low Res', disabled=False, layout=wg.Layout(width='100px'))
close_ts_butt       = wg.Button(description="Close Graph", icon = "times")
clear_list_butt     = wg.Button(description="Clear List", icon = "times")
ts_sites            = wg.SelectMultiple(options=[], value=[], description='Site List:')

# ts customizations

error_bar_check     = wg.Checkbox(description="Error Bars", value=False)
error_bar_outline_check = wg.Checkbox(description = "Error Bar Outlines", value = False)

detrend_check       = wg.Checkbox(description="Detrend", value=False)
live_update_check   = wg.Checkbox(description="Live Update", value=False)
start_year_form     = wg.Text(value=None,placeholder='YYYY-MM-DD',description='Start:',disabled=False, style= {'description_width': 'initial'}, layout=layout("150px"))
end_year_form       = wg.Text(value=None,placeholder='YYYY-MM-DD',description='End:',disabled=False, style= {'description_width': 'initial'}, layout=layout("150px"))
siglim_form         = wg.Text(value=None,placeholder='(10,10,30)',description='SigLim:',disabled=False, style= {'description_width': 'initial'}, layout=layout("150px"))

shift_value        = wg.FloatText(value=None,placeholder='0',description="Shift Values (mm)",disabled=False, style= {'description_width': 'initial'}, layout=layout("200px"))
remove_site_button  = wg.Button(description="Remove Site", icon = "minus-square")
update_customization = wg.Button(description="Update shift", icon = "arrow-up")
shift_output       = wg.Output()

thin_error_bars    = wg.IntText(description = "NErrBar", value = 1, layout=layout("150px"),tooltip='Show every Nth error bar')
error_bar_opacity  = wg.FloatText(description = "EB Opacity", layout=layout("150px"), value = 0.1)
add_breaks_ts_button = wg.Button(description = "Copy Break Data")
remove_breaks_ts_button = wg.Button(description="Clear Break Data")

remove_outliers = wg.Dropdown(description = "Remove N σ", options=[None, 1, 2, 3, 4, 5, 6], value = None, disabled = False, layout=wg.Layout(width='150px'))
show_breaks_data_button = wg.Button(description = "Show Breaks Data")
remove_breaks_checkbox = wg.Checkbox(description = "Remove Breaks", value = False)
breaks_data_output = wg.Output()

# ID points option (only in iterative mode)
id_button =  wg.Button(description = "ID points",tooltip='right-click/return to end\nmiddle/delete to remove')

# Style
append_butt.style.button_color = 'rgb(196,253,196)'
plot_ts_butt.style.button_color = 'paleturquoise'
close_ts_butt.style.button_color = 'mistyrose'
clear_list_butt.style.button_color = 'mistyrose'
remove_site_button.style.button_color = 'mistyrose'
id_button.style.button_color = 'springgreen'

# Graphics Backends
backend_butt = wg.RadioButtons(
    options=['Inline', 'ipympl', 'Interactive'],
    value='Inline',
    description='Backend Choice:',
    disabled=False
)
backend_activate = wg.Button(description = "Backend Activate")

# Outputs
timeseries_output   = wg.Output()

### Widget Functions

#### Database Fetcher

In [ ]:
def UpdateUNR(_):
    with fetcher_output:
        print("Downloading Data from UNR...")

    # = "https://geodesy.unr.edu/gps_timeseries/sta_frames.txt"
    # Updated to file location URL
    plates_site = "https://geodesy.unr.edu/gps_timeseries/Plates/sta_frames.txt"
    coords_site = "https://geodesy.unr.edu/NGLStationPages/llh.out"
    #vels_site   = "https://geodesy.unr.edu/velocities/midas.IGS08.txt"
    # Updated to IGS14 TAH 260330.
    vels_site   = "https://geodesy.unr.edu/velocities/midas.IGS14.txt"

    plates_list = str(urllib.request.urlopen(plates_site).read())[2:].split(" \\n")[:-1]
    coords_list = str(urllib.request.urlopen(coords_site).read())[2:].split(" \\n")[:-1]

    for_json = {}

    for siteplate in plates_list:
        siteplates = [el for el in siteplate.split(" ") if el]
        for_json.setdefault(siteplates[0], {'location': None, 'height': [], 'regions': []})
        for_json[siteplates[0]]['regions'] = siteplates[1:]

    for sitecoord in coords_list:
        siteid, lon, lat, height = (el for el in sitecoord.split(" ") if el)
        for_json.setdefault(siteid, {'location': None, 'height': [], 'regions': []})
        for_json[siteid]['location'] = [float(lon), float(lat)]
        for_json[siteid]['height'] = float(height)

    # MOD TAH: Add reading UNR velocities 
    df = pd.read_csv(vels_site,delimiter=r"\s+",usecols=[0,8,9,10,11,12,13])
    nvel = len(df) ; print("Processing",nvel,"lines")
    for i in range(nvel):
        siteid = df.iloc[i,0]
        for_json.setdefault(siteid, {})
        for_json[siteid]["velocity"] = list(df.iloc[i,[2,1,3]]*1000) # switch to mm and from ENU to NEU
        for_json[siteid]["velsig"] = list(df.iloc[i,[5,4,6]]*1000)
        # if ( i < 10 ):
        #     print('Input i',i,df.iloc[i,[2,1,3]]*1000)
        #     print("Vel for ",siteid,for_json[siteid]["velocity"])
     

    global data_of
    data_of["UNR"] = for_json

    with open('./data/UNR/UNR_data.json', 'w') as f:
        json.dump(for_json, f)

    with fetcher_output:
        print("Latest UNR Data Downloaded")

update_UNR_butt.on_click(UpdateUNR)


def UpdateJPL(_):
    with fetcher_output:
        print("Downloading Data from JPL...")

    coords_site = "https://sideshow.jpl.nasa.gov/post/tables/table2.html"
    x = str(urllib.request.urlopen(coords_site).read()).split("\\n")
    sitecoordlist = [[xxx for xxx in xx.split(" ") if xxx]
                     for xx in x if "POS" in xx]
    sitevellist = [[xxx for xxx in xx.split(" ") if xxx]
                     for xx in x if "VEL" in xx]
    # [['AB01', 'POS', '52.2095', '-174.2048', '25492.217', '0.034', '0.024', '0.098'], ...]
    for_json = {}
    for element in sitecoordlist:
        for_json[element[0]] = {'location': [float(element[2]),float(element[3])],'height': float(element[4])/1000.0}
    for element in sitevellist:
        for_json[element[0]]['velocity'] = [float(element[2]),float(element[3]), float(element[4])]
        for_json[element[0]]['velsig'] = [float(element[5]),float(element[6]), float(element[7])]

    global data_of
    data_of["JPL"] = for_json

    with open('./data/JPL/JPL_data.json', 'w') as f:
        json.dump(for_json, f)

    with fetcher_output:
        print("Latest JPL Data Downloaded")
update_JPL_butt.on_click(UpdateJPL)


def UpdateNGF(_):
    with fetcher_output:
        print("Downloading Data from NGF...")

    coords_site = "https://www.unavco.org/instrumentation/networks/status/data/geoJSON/network-monitoring"

    x = json.loads(urllib.request.urlopen(coords_site).read())
    print('Back from json.load ')
    # [['AB01', 'POS', '52.2095', '-174.2048', '25492.217', '0.034', '0.024', '0.098'], ...]
    for_json = {}
    for element in x['features']:
        # print("NGF",element)
        for_json[element["id"]] = {
            'location': [element['geometry']['coordinates'][1],element['geometry']['coordinates'][0]],
            'height': float(element['properties']['elev']),
            'region': element['properties']['region'],
            'stntype': element['properties']['stntype']}

    # Read Breaks #
    if ( esver[0] == 0 ):
        get_es_file("https://gage-data.earthscope.org/archive/gnss/products/offset/cwu.kalts_nam14.off")
    else:
        try:
            from earthscope_sdk import EarthScopeClient
            from earthscope_sdk.auth.auth_flow import NoAccessTokenError
            es_client = EarthScopeClient()
            token = es_client.ctx.auth_flow.access_token
            headers = {"Authorization": f"Bearer {token}"}
        except NoAccessTokenError:
            print("EarthScope authentication required.")
            print("Run  `es login`  in your terminal, then re-run this cell.(First time? Run `pip install earthscope-cli` first.)")
            return
        fname = "cwu.kalts_nam14.off"
        outpath = Path("./data/NGF/cwu.kalts_nam14.off")
        url = "https://gage-data.earthscope.org/archive/gnss/products/offset/cwu.kalts_nam14.off"
        print('Calling fetch_csv with ',url,fname)
        url, ok, err = fetch_csv(url, outpath, headers=headers)
        if ok:
            print(f"✓ {fname}")
        else:
            print(f"✗ {fname}  --  {err}")
            #failures.append((fname, err))

    with open("./data/NGF/cwu.kalts_nam14.off","r") as f:
        data = list(f)

    i = 0
    while 'End Field Description' not in data[i]: i += 1

    data = data[i+1:]

    for i in range(1, len(data)):
        info = data[i].split()
        siteid = info[0]
        for_json.setdefault(siteid, {})
        dt = [int(pt) for pt in info[1:6]]
        moment = str(tuple(dt[0:5]))
        for_json[siteid].setdefault("breaks", {})
        for_json[siteid]["breaks"][moment] = {"offsets": [float(pt) for pt in info[6:12]],
                                              "type": info[12],
                                              "description": " ".join(info[13:]),}
    ###############

    # Read Velocities MOD TAH: Replace final with snaps file.
    for velf in ('cwu.snaps_igs14.vel','cwu.fanet_igs14.vel') : 
        if ( esver[0] == 0 ):
            get_es_file("https://gage-data.earthscope.org/archive/gnss/products/velocity/"+velf)
        # https://gage-data.earthscope.org/archive/gnss/products/velocity/cwu.fanet_igs14.vel
        else :
            fname = velf
            outpath = Path("./data/NGF/"+velf)
            url = "https://gage-data.earthscope.org/archive/gnss/products/velocity/"+velf
            print('Calling fetch_csv with ',url,fname)
            url, ok, err = fetch_csv(url, outpath, headers=headers)
            if ok:
                print(f"✓ {fname}")
            else:
                print(f"✗ {fname}  --  {err}")
                #failures.append((fname, err))


        with open("./data/NGF/"+velf,"r") as f:
            data = list(f)

        i = 0
        while 'End Field Description' not in data[i]: i += 1

        data = data[i+1:]

        for i in range(1, len(data)):
            info = data[i].split()
            siteid = info[0]
            for_json.setdefault(siteid, {})
            for_json[siteid]["velocity"] = [float(pt)*1000 for pt in info[19:22]] # switch to mm
            for_json[siteid]["velsig"] = [float(pt)*1000 for pt in info[22:25]]
            # Get positions from velocity file 8,9,10 lat, long, height
            if( float(info[8])>180 ):
                for_json[siteid]["location"] = [float(info[7]),float(info[8])-360]
            else:
                for_json[siteid]["location"] = [float(info[7]),float(info[8])]

            for_json[siteid]["height"] = float(info[9])


    ###############

    global data_of
    data_of["NGF"] = for_json

    with open('./data/NGF/NGF_data.json', 'w') as f:
        json.dump(for_json, f)

    with fetcher_output:
        print("Latest NGF Data Downloaded")
update_NGF_butt.on_click(UpdateNGF)


#### Availability

In [ ]:
def site_search(b):
    searched = site_searchbar.value.upper()
    with availability_output:
        try:
            print(searched + "(" + org_avail_select.value + ")", ": ")
            org = org_avail_select.value
            long_tuple = data_of[org][searched]
            long_string = str(long_tuple)
            # Wrap the string to a maximum width of 70 characters
            wrapped_string = textwrap.wrap(long_string, width=150)
            # Print each wrapped line
            for line in wrapped_string:
                print(line)
        except:
            display(wg.HTML('''<em style="color:red">Not Found!</em>'''))

site_search_submit.on_click(site_search)


def clear_avail_log(b):
    availability_output.clear_output()
clear_log_butt.on_click(clear_avail_log)

def add_breaks_data_from_json(b):
    org = breaks_select.value
    for site in data_of[org]:
        if site in data_of["NGF"]  and "breaks" in data_of["NGF"][site]:
            data_of[org][site]["breaks"] = data_of["NGF"][site]["breaks"]
    with breaks_update_output:
        display(f"Breaks added to {org} data")
    print(f"Breaks added to {org} data")

add_breaks_site_button.on_click(add_breaks_data_from_json)

def add_indv_breaks_data_from_NGF(b):
    selected = list(ts_sites.value)
    for site in selected:
        siteid, org = remove_brac(site) # remove brac takes "P234 (UNR)" and returns ("P234", "UNR")
        if siteid in data_of["NGF"]:
            data_of[org][siteid]["breaks"] = data_of["NGF"][siteid]["breaks"]
    with breaks_data_output:
        display(f"NGF breaks data added to {siteid} ({org}) data")

add_breaks_ts_button.on_click(add_indv_breaks_data_from_NGF)

def remove_breaks_data_from_json(b):
    org = breaks_select.value
    for site in data_of[org]:
        try:
            del data_of[org][site]["breaks"]
        except:
            pass
    with breaks_update_output:
        display(f"Breaks removed from {org} data")

remove_breaks_site_button.on_click(remove_breaks_data_from_json)

def remove_indv_breaks_data_from_NGF(b):
    selected = list(ts_sites.value)
    for site in selected:
        siteid, org = remove_brac(site) # remove brac takes "P234 (UNR)" and returns ("P234", "UNR")
        try:
            del data_of[org][siteid]["breaks"]
        except:
            pass
        display(f"NGF breaks data removed from {siteid} ({org}) data")
remove_breaks_ts_button.on_click(remove_indv_breaks_data_from_NGF)

#### Map

In [ ]:
# Map Functions
# Choices of maps come from https://leaflet-extras.github.io/leaflet-providers/preview/
# Satellite image:
#var Stadia_AlidadeSatellite = L.tileLayer('https://tiles.stadiamaps.com/tiles/alidade_satellite/{z}/{x}/{y}{r}.{ext}', {
#	attribution: '&copy; CNES, Distribution Airbus DS, © Airbus DS, © PlanetObserver (Contains Copernicus Data) | &copy; <a href="https://www.stadiamaps.com/" target="_blank">Stadia Maps</a> &copy; <a href="https://openmaptiles.org/" target="_blank">OpenMapTiles</a> &copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors',
#	ext: 'jpg'});
#
# var Esri_WorldImagery = L.tileLayer('https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}', {
#	attribution: 'Tiles &copy; Esri &mdash; Source: Esri, i-cubed, USDA, USGS, AEX, GeoEye, Getmapping, Aerogrid, IGN, IGP, UPR-EGP, and the GIS User Community'
#});
def BaseMap_Set(b):
    # Function to set tile and attr for basemap
    attr = (
        '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> '
        'contributors, &copy; <a href="https://cartodb.com/attributions">CartoDB</a>'
        'contributors, <a href="http://viewfinderpanoramas.org">SRTM</a>' 
        ) 
    if (BaseMap_butt.value == 'OpenTopo' ) :
        tiles = "https://{s}.tile.opentopomap.org/{z}/{x}/{y}.png"
    elif ( BaseMap_butt.value == 'OpenStreetMap') :
        tiles = "https://tile.openstreetmap.org/{z}/{x}/{y}.png"
    else:
        tiles = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"
        attr = ('Tiles &copy; Esri &mdash; Source: Esri, i-cubed, USDA, USGS, AEX, GeoEye, Getmapping, Aerogrid, IGN, IGP, UPR-EGP, and the GIS User Community')

    return tiles, attr


#tiles = "https://{s}.tile.opentopomap.org/{z}/{x}/{y}.png"
#tiles = "https://tile.openstreetmap.org/{z}/{x}/{y}.png"
#tiles = "https://tiles.stadiamaps.com/tiles/alidade_satellite/{z}/{x}/{y}{s}.jpg"
#tiles = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"
#var OpenTopoMap = L.tileLayer(, {
#	maxZoom: 17,#
#0  attribution: 'Map data: &copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors, <a href="http://viewfinderpanoramas.org">SRTM</a> | Map style: &copy; <a href="https://opentopomap.org">OpenTopoMap</a> (<a href="https://creativecommons.org/licenses/by-sa/3.0/">CC-BY-SA</a>)'
#});

#folium.Map(location=[lat, lon], tiles=tiles, attr=attr, zoom_start=zoom_start)

def new_map(b):
    map_output.clear_output()
    # See if we have map point set
    [tiles,attr] = BaseMap_Set(b)

    ll = (40,-100) ; zoom = 4  
    for option in list(map_radius_list.value):
        option = option.split(", ")
        ll = data_of[option[1]][option[0]]["location"]
        zoom = 6
        # selected_options[option[0]] = {"org":option[1], "radius":option[2]}
    # print('Center',ll,'Zoom',zoom)

    map[0] = fol.Map(control_scale=True,
                   location=ll,
                   zoom_start=zoom, tiles=tiles, attr=attr,
                   width = 1000,
                   height = 600)
    with map_output:
        display(map[0])
new_map_butt.on_click(new_map)

def reload_map(b):
    map_output.clear_output()
    with map_output:
        display(map[0])

reload_map_butt.on_click(reload_map)

def close_map(b):
    map_output.clear_output()
    nearest_site_output.clear_output()
    graph_output.clear_output()
    map[0] = "Not initialized yet!"
close_map_butt.on_click(close_map)


####################
# SITE/RADIUS LIST #
####################

def update_map_list(b):
    siteid = site_id.value
    radius = site_radius.value
    org = org_map_select.value
    if siteid in data_of[org].keys():
        addition = f"{siteid}, {org}, {radius}km"
        map_radius_list.options = list(map_radius_list.options) + [addition]
    else:
        print("Not A Valid Site!")

    # MOD TAH 241223: Don't clear site name
    #site_id.value = ""   

add_sites_map_button.on_click(update_map_list)

def remove_map_list(b):
    options = list(map_radius_list.options)
    for option in map_radius_list.value:
        options.remove(option)
    map_radius_list.options = options

remove_sites_map_button.on_click(remove_map_list)

################
# SITE BUTTONS #
################

# def plot_locations(map, coordslist, tooltiplist, color="blue", dotrad = 2):

def site_circle(b):
    new_map(b)

    selected_options = {}

    for option in list(map_radius_list.value):
        option = option.split(", ")
        selected_options[option[0]] = {"org":option[1], "radius":option[2]}

    # siteid, rad = site_id.value, site_radius.value
    plotvec = plot_vec_check.value

    # [('P036', 36.420274, -105.293656, 60.19983176958408), ('RG11', 36.523165, -105.779104, 17.132432998877896)] [36.635171, -105.91085]
    # [(36.420274, -105.293656), (36.523165, -105.779104)] ['P036', 'RG11']

    for siteid in selected_options:
        org = selected_options[siteid]["org"]
        radius = int(selected_options[siteid]["radius"][0:-2])

        plot_locations(map[0],
        [data_of[org][siteid]["location"] if isinstance(siteid, str) else siteid],
        [f"{siteid}, no vel. data"] if not isinstance(get_vel(siteid, org), tuple) else [f"{siteid}, vel:{get_vel(siteid, org)[0]}"],
        color="black", dotrad = 4)

        if plotvec:
            plot_velocity(siteid, org, list(eval(vel_siglim_form.value)) if vel_siglim_form.value else [1,1,2])

        if radius > 0:
            site_list, center = nearby_sites(org, siteid, radius)
            basic_circle(map[0], center, radius)
            site_name_list = [el[0] for el in site_list]  # next few lines check if there's a velocity, then make it show up in tooltip if yes
            tooltip_list = []
            for site in site_name_list:
                if isinstance(get_vel(site, org), tuple):
                    # [f"{int(x*100)/100}" for x in get_vel('KAGA', 'NGF')[0]]
                    #tooltip_list.append(f"{site}, vel:{get_vel(site, org)[0]}")
                    velstr = [f"{x:.2f}" for x in get_vel(site, org)[0]] ; 
                    velflt = [float(x) for x in velstr]
                    ll = data_of[org][site]['location'] ; ht = data_of[org][site]['height']
                    locstr = f"{ll[0]:.5f} {ll[1]:.5f} {ht:.3f}"
                    allstr = f"{site}, vel: {velflt}, loc: {locstr}"
                    #print(allstr) #                  
                    tooltip_list.append(allstr)

                else:
                    tooltip_list.append(f"{site}, no vel. data")

            plot_locations(map[0], [el[1:3] for el in site_list], tooltip_list)

            if plotvec:
                counter = 0
                for el in site_list: # incredment a counter and plot all sites that match some number (that is taken from an input), then reset the counter
                    if counter == 0:
                        plot_velocity(el[0], org, list(eval(vel_siglim_form.value)) if vel_siglim_form.value else [1,1,2])
                    counter += 1
                    if counter == thin_velocities_input.value:
                        counter = 0

                # Now put a velocity scale on the plot (put 1.5*radius below reference site)
                vsloc = np.copy(data_of[org][siteid]["location"])
                vsrad = site_radius.value
                vsloc[0]=vsloc[0]+1.1*vsrad/111.0 # Put 1.1 radius above center
                vsvel = [0, arrow_length_input.value, 0]
                vslab = f"{vsvel[1]:.0f} mm/yr"
                #vsvel = [0, 10, 0]
                draw_vector(map[0], vsloc, vsvel, velocity_scale_input.value, color="black",label=vslab) 
        

    # if siteid[0] in "[(":
    #     siteid = list(eval(siteid))

    reload_map(b)

    # Skip code to show color bar for vertical signals
    # if plotvec:
    #     with map_output:
    #         fig, ax = plt.subplots(figsize=(6,0.5), dpi = 150 )
    #         xmax = 255/colorscalefactor
    #         for x in range(-255, 256):
    #             ax.axvline(x/colorscalefactor, color=(max(x/255,0),0,max(-x/255,0)), linewidth=4)
    #         ax.get_yaxis().set_ticks([])
    #         ax.set_xlim([-xmax, xmax])
    #         ax.set_xlabel("ΔUp Velocity (mm/yr)")
    #         plt.show()

site_circle_submit.on_click(site_circle)


def nn_graph(b):
    nearest_site_output.clear_output()
    graph_output.clear_output()
    selected_options = {}
    for option in list(map_radius_list.value):
        option = option.split(", ")
        selected_options[option[0]] = {"org":option[1], "radius":int(option[2][0:-2])}


    for site in selected_options:
        if selected_options[site]["radius"] > 0:
            siteid = site
            rad = selected_options[site]["radius"]
            org = selected_options[site]["org"]
            site_list, _ = nearby_sites(org, siteid, rad)
            neighbours = sorted(site_list, key = lambda x: x[3])[:10] # sorted list of 10 nearest ("Site", lat coord, long coord, distance from site)
            velocities = [] ; relvel = [] ;
            # get reference site velocity
            refvel = data_of[org][siteid]["velocity"]

            for site in neighbours:
                if isinstance(get_vel(site[0], org), tuple):
                    vel = get_vel(site[0], org)[0]                   
                    velocities.append(get_vel(site[0], org))
                    relvel.append(vec_sub(vel,refvel))
                else:
                    velocities.append(("None","None"))
                    relvel.append("None")
                                   
            with nearest_site_output:

                display(wg.HTML("""<h2>10 Nearest Neighbouring Sites:</h2>"""))
                tableHTML = f"<table><tr><th><b>Site</b></th><th><b>Distance from {siteid}</b></th><th><b>Velocity (n, e, u)</b></th><th><b>VelSig (n, e, u)</b></th><th><b>RelVel (n, e, u)</b></th></tr>"
 #              for i in range(len(neighbours)): tableHTML += f"<tr><td>{neighbours[i][0]}</td><td>{neighbours[i][3]:.3f} km</td><td>{velocities[i][0]}</td><td>{velocities[i][1]}</td></tr>"
                for i in range(len(neighbours)):
                    if ( velocities[i][0] != "None") :
                        velstr = [f"{x:.2f}" for x in velocities[i][0]] ; velflt = [float(x) for x in velstr]
                        sigstr = [f"{x:.2f}" for x in velocities[i][1]] ; sigflt = [float(x) for x in sigstr]
                        dvelstr =  [f"{x:.2f}" for x in relvel[i]] ; dvelflt = [float(x) for x in dvelstr]
                    else:
                        velstr = "None" ; sigstr = "None" ; dvelstr = "None"
                    
                    tableHTML += f"<tr><td>{neighbours[i][0]}</td><td>{neighbours[i][3]:.3f} km</td><td>{velflt}</td><td>{sigflt}</td><td>{dvelflt}</td></tr>"
                display(wg.HTML(tableHTML + "</table>"))

            # See if inline backend (if not create separate figure)
            s = str(plt.get_backend())
            indx = s.find('inline')
            if( indx > 0 ): 
                with graph_output:
                    display(wg.HTML("""<h2 style="text-align:center;">Distances to Nearby Sites (km)</h2>"""))
                    plt.bar([x[0] for x in neighbours], [x[3] for x in neighbours])
                    plt.show()
            else:
                print('Creating new figure backend: ',plt.get_backend)
                plt.figure(figsize=(8, 6))
                plt.bar([x[0] for x in neighbours], [x[3] for x in neighbours])
                plt.show()
                 

    # {"AB01": {"location": [52.209501, -174.204758], "velocity": [-23.472, -7.111, 1.69], "velsig": [0.003, 0.002, 0.009]}

    # if not rad:
    #     nearest_site_output.clear_output()
    #     graph_output.clear_output()
    #     return



plot_graph_butt.on_click(nn_graph)


def get_vel(siteid, org):
    # jpl format {"AB01": {"location": [52.209501, -174.204758], "velocity": [-23.472, -7.111, 1.69], "velsig": [0.003, 0.002, 0.009]}
    try:
        return data_of[org][siteid]["velocity"], data_of[org][siteid]["velsig"]
    except:
        return None

def plot_velocity(siteid, org, siglim):
    if get_vel(siteid, org): velocity, velsig = get_vel(siteid, org)
    else: return

    # See if plotting relative velocity
    if( plot_rel_vel.value ) :
        # Get velocity of reference site
        for option in list(map_radius_list.value):
            option = option.split(", ")
            site_ref = option[0]
            refvel = data_of[org][site_ref]["velocity"]

    else:
        refvel = [0,0,0]
    # Only update N and E values (leave height unchanged)
    pltvel = [velocity[0]-refvel[0], velocity[1]-refvel[1], velocity[2]]

        
    # data_of[org][siteid]["velocity"] = velocity
    # -10 <=> rgb(0,0,255)
    # 0 <=> rgb(0,0,0)
    # 10 <=> rgb(255,0,0)
    # Get Up velocity scaling factor ±255/colorscalefactor
    # Convert from ±<Range> to colorfactor: Red down; Blue Up
    colorscalefactor = 255/colorscalefactor_input.value 

    if all(velsig[i] <= siglim[i] for i in range(3)):
        zv = velocity[2]
        if zv > 0:
            rv, bv = 0, min(255, zv * colorscalefactor)
        else:
            rv, bv = min(255, zv * ( -colorscalefactor)), 0 
        #print('PLotting',siteid,pltvel,'Ref ',refvel)
        draw_vector(map[0], data_of[org][siteid]["location"], pltvel, velocity_scale_input.value, color="rgb(" + str(rv) + ",0," + str(bv) + ")")


# enter_code_butt.on_click(plot_velocity)

def draw_length(x):
    length = arrow_length_input.value
    coords = length_coords_text.value.split(",")
    for i in range(len(coords)):
        coords[i] = float(coords[i])

    end_coords = calc_distance_earth(coords[0], coords[1], length)
        
    fol.PolyLine(locations = [coords, end_coords],
        weight = 3,
        tooltip = f"{length} mm/yr"
    ).add_to(map[0])
    with map_output:
        map_output.clear_output()
        display(map[0])

#draw_length_button.on_click(draw_length)

def clear_map_list(b):
    map_radius_list.options = []
clear_map_list_butt.on_click(clear_map_list)


#### Timeseries

In [ ]:
def update_ts_list(x):
    siteid = ts_site_form.value

    # check validity
    if siteid not in data_of[org_ts_select.value]:
        timeseries_output.clear_output()
        with timeseries_output:
            display(wg.HTML('''<em style="color:red">Site not in ''' + org_ts_select.value + '''.</em>'''))
    else:
        extendedsiteid = siteid + " (" + org_ts_select.value + ")"
        if extendedsiteid not in ts_sites.options:
            ts_sites.options = list(ts_sites.options) + [extendedsiteid]

    #ts_site_form.value = ""

append_butt.on_click(update_ts_list)

def list_to_graph(x):
    global tsfig
    selected = list(ts_sites.value)
    timeseries_output.clear_output()

    # checked for shift values
    if len(shift_dict_all) != 0:
        shift_dict_selected = {}
        for site in selected:
            if site in shift_dict_all:
                shift_dict_selected[site] = shift_dict_all[site]
    else:
        shift_dict_selected = {}


    if siglim_form.value:
        global SigLim
        try:
            nlim, elim, ulim = eval(siglim_form.value)
            assert all(isinstance(lim,int) for lim in [nlim, elim, ulim])
        except:
            nlim, elim, ulim = 10, 10, 30
        SigLim = (nlim, elim, ulim)

    # check validity

    yearrange = [datetime(1,1,1), datetime(3000,1,1)] # [far past, far future]

    if selected:
        if start_year_form.value: yearrange[0] = datetime.strptime(start_year_form.value, "%Y-%m-%d")
        if end_year_form.value: yearrange[1] = datetime.strptime(end_year_form.value, "%Y-%m-%d")

        with timeseries_output:
            plot_ts_graph_list(
                selected,
                int(live_update_check.value),
                yearrange,
                breaks = remove_breaks_checkbox.value,
                detrend = detrend_check.value,
                errorbars = [error_bar_check.value, thin_error_bars.value],
                errorbar_outline = [error_bar_outline_check.value, error_bar_opacity.value],
                outlier = remove_outliers.value,
                resolution = plot_ts_res.value,
                shift = shift_dict_selected,
                )
    else:
        with timeseries_output:
            display(wg.HTML('''<em style="color:red">Select a set of sites to plot!</em>'''))

    # ts_site_form.value = ""

plot_ts_butt.on_click(list_to_graph)

###

def remove_site(x):
    options = list(ts_sites.options)
    for site in ts_sites.value:
        options.remove(site)
    ts_sites.options = options

remove_site_button.on_click(remove_site)

shift_dict_all = {}

def add_shift(x):
    selected = list(ts_sites.value)
    shift_output.clear_output()
    if shift_value.value:
        for site in selected:
            shift_dict_all[site] = shift_value.value
    else:
        for site in selected:
            del shift_dict_all[site]

    with shift_output:
            print("shifts:")
            print(shift_dict_all)

update_customization.on_click(add_shift)

# def clear_shifts(b):

def display_break_data(x):
    breaks_data_output.clear_output()

    for siteid in ts_sites.value:
        site, org = remove_brac(siteid)
        if "breaks" in data_of[org][site]:
            with breaks_data_output:
                display(wg.HTML(f"{siteid} Break Data"))
                tableHTML = f"<table><tr><th><b>Date (yr, m, d, hr, min)</b></th><th><b>Offset in mm (dN, sN, dE, sE, dU, sU)</b></th><th><b>Type</b></th><th><b>Description</b></th></tr>"
                for i in data_of[org][site]["breaks"]:
                    tableHTML += f'<tr><td>{i}</td><td>{data_of[org][site]["breaks"][i]["offsets"]}</td><td>{data_of[org][site]["breaks"][i]["type"]}</td><td>{data_of[org][site]["breaks"][i]["description"]}</td></tr>'
                display(wg.HTML(tableHTML + "</table>"))
        else:
            with breaks_data_output:
                display(f"{siteid}: No Breaks Data")

show_breaks_data_button.on_click(display_break_data)

### Backend activate function
def backend_act(b):
    # Seems to be needed to get interactive graphic to appear
    systype = sys.platform
    if( backend_butt.value == 'Inline') :
        #mpl.use('inline',force=True)
        # %matplotlib notebook
        try:
            %matplotlib inline
        except:
            with timeseries_output:
                print('Problem selecting inline')
    elif ( backend_butt.value == 'ipympl') :
        #mpl.use('ipympl',force=True)
        try:
            %matplotlib ipympl
        except:
            with timeseries_output:
                print('Problem with selecting ipympl backend')

    else :
        print('Checking System',systype)
        if (systype=='darwin' ) :
            try: 
                %matplotlib osx
            except:
                with timeseries_output:
                    print('Problem with osx with ',systype)
    
            # mpl.use('macosx')
        elif (systype == 'posix' or systype == 'linux'):
            try:
                %matplotlib qt5agg
            except:
                with timeseries_output:
                    print('Problem with qt5agg with ',systype)
            
        else :
            try: 
                %matplotlib qt
            except:
                with timeseries_output:
                    print('Problem with qt with ',systype)
                    
    with timeseries_output:
        print('Using backend ',mpl.get_backend(),systype)


backend_activate.on_click(backend_act)    


def close_ts(b):
    timeseries_output.clear_output()
close_ts_butt.on_click(close_ts)

def clear_list(b):
    ts_sites.options = []
clear_list_butt.on_click(clear_list)

def id_points(b):
    global tsfig
    #username = os.getlogin()
    username = os.environ.get('USER', os.environ.get('USERNAME'))
    if ( backend_butt.value == 'Interactive') :
        pts = tsfig.ginput(n=10, show_clicks=True)
        #print('Selected point ',pts)
        cnt = 0 
        for pt in pts :
            cnt += 1
            dt = pt[0]*86400
            epoch = datetime.fromtimestamp(dt,tz=timezone.utc).strftime("%Y %m %d %H %M")
            ptstr = f"Point {cnt:2d} Epoch {epoch} Value {pt[1]:.2f} mm"
            with timeseries_output:
                print(ptstr)

        cnt = 0
        for pt in pts :
            dt = pt[0]*86400
            epoch = datetime.fromtimestamp(dt,tz=timezone.utc).strftime("%Y %m %d %H %M")
            #np.where(times >= breaktime)[0][0]
            ax= tsfig.axes
            ax[2].axvline(x=pt[0], color='g', linestyle='dotted', linewidth=2)
            ax[1].axvline(x=pt[0], color='g', linestyle='dotted', linewidth=2)
            ax[0].axvline(x=pt[0], color='g', linestyle='dotted', linewidth=2)
            with timeseries_output:
                siteid = ts_sites.value[0][0:4]
                code = "_"+chr(ord('A') + cnt)+"PS"
                current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                # print('Point ',epoch,'Days ',pt[0],' Value ',pt[1])
                print(" rename",siteid,"   ",siteid+code,epoch,"                  ! GNSS_Analysis",username,current_time)
            cnt += 1


    else:
        with timeseries_output:
            display(wg.HTML('''<em style="color:red">ID only works with Interactive Backend activated'!</em>'''))

id_button.on_click(id_points)


### Windows Layout

In [ ]:
# Windows Layout
fetch_window = VBox([
    wg.HTML(
            '<em style="color:blue; line-height: 1.5">Use the buttons below to update the local files with the latest data. This may take a few seconds.</em>'
            ),
    HBox([update_NGF_butt, update_UNR_butt, update_JPL_butt]),
    fetcher_output,
    HBox([breaks_select, add_breaks_site_button, remove_breaks_site_button]),
    HBox([breaks_update_output])
])

availability_window = VBox([
    wg.HTML('<h1>Availability</h1>'),
    wg.HTML(
            '<em style="color:blue; line-height: 1.5"><ul>' +
            '<li>Type the Site ID into the box below to check its existence and status.</li>' +
            '</ul></em>'
            ),
    HBox([site_searchbar, org_avail_select, site_search_submit, clear_log_butt]),
    availability_output
    ])

# _style = HTML(
#     "<style>.widget-radio-box {flex-direction: row !important;}.widget-radio-box"
#     " label{margin:5px !important;width: 200px !important;}</style>",
#     layout=Layout(display="none"),
# )

map_window = VBox([
    wg.HTML('<h1>Map</h1>'),
    HBox([new_map_butt, reload_map_butt, close_map_butt]),
    HBox([site_id, org_map_select, site_radius, add_sites_map_button]),
    HBox([map_radius_list, site_circle_submit, remove_sites_map_button, clear_map_list_butt]),
    HBox([vel_siglim_form, plot_graph_butt]),
    HBox([thin_velocities_input,  plot_vec_check, plot_rel_vel]),
    #HBox([arrow_length_input, length_coords_text, draw_length_button]),
    HBox([arrow_length_input, velocity_scale_input,colorscalefactor_input]),
    HBox([BaseMap_butt]),
    HBox([nearest_site_output, graph_output]),
    map_output,
    ])

timeseries_window = VBox([
    wg.HTML('<h1>Timeseries</h1>'),
    HBox([ts_site_form, org_ts_select, append_butt, live_update_check]),
    HBox([ts_sites, VBox
          ([HBox([plot_ts_butt, show_breaks_data_button]),
            HBox([remove_site_button, clear_list_butt]),
            HBox([close_ts_butt,id_button])])
        ]),
    HBox([start_year_form, end_year_form, siglim_form, remove_outliers]),
    HBox([remove_breaks_checkbox, detrend_check]),
    HBox([error_bar_check, error_bar_outline_check]),
    HBox([error_bar_opacity, thin_error_bars, add_breaks_ts_button, remove_breaks_ts_button]),
    HBox([shift_value, update_customization]),
    HBox([shift_output]),    
    HBox([backend_butt,backend_activate])
])
timeseries_output_window = VBox([breaks_data_output, timeseries_output])

In [ ]:
print("Running on the following verions:")
print(f"folium: {fol.__version__}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"ipywidgets: {wg.__version__}")
print(f"matplotlib: {mpl.__version__}")
print(f"urllib.request: {urllib.request.__version__}")
print(f"requests: {requests.__version__}")

Test on the following versions:

- folium: 0.14.0
- pandas: 2.2.1
- numpy: 1.26.4
- ipywidgets: 8.1.2
- matplotlib: 3.8.4
- urllib.request: 3.11
- requests: 2.31.0

## Interface and Downloads

To use the program for the first time, you must run the fetch_window display and click the "Latest" button for every source you want data for. This will download the data of all sites for the source (excluding time-series data) from the relevant website into the "data" folder. Upon using this program again, you may skip this step if you don't want to update the data from the web. The data, once downloaded, will show up in the data/"source" folder as "site"_data.json. 

## Availability

The "Check Availability" box allows you to if data exists for a site in a given source. It will directly output the data for that site as it is stored in the JSON file (in the form of a dictionary). 

Adding/Remove Breaks: If for all sites in the selected organization, if the site is also available in the NGF data, it will take the NGF break data and add it to the non-NGF version of the site, allowing for the "remove breaks" and "show breaks data" button to be available for the graphing portion of the notebook. A way to do this for individual sites can be found in the time-series section below. 



In [ ]:
display(fetch_window, availability_window) # run to display fetcher and availability options

## Using the Map

Clear/New Map, Reload Map, Close Map: Self Explanatory

Site ID + Source Dropdown + Radius + Add Sites: Inputting the combination of a valid site id, a radius, selecting a source, and then clicking "Add Sites" will add the format of "Siteid (Source) Radius km" to the Site/Radius Box. 

Site/Radius Box + Add Plot!: Here, select as many options as wished to be added to the map with the "Add Plot!" Button. This will show the site as well as all other sites within the selected radius. Other functions will also only work if the relevant site is selected from this box. 

Remove Sites: This will remove the Site/Source/Radius combination(s) from the Site/Radius Box

Clear All: Clears all sites from the Site/Radius box

Distance Table: This will show a table with the distances of the 10 nearest neighboring sites to the center site. 
It will also display their velocities, velocity relative to the first site, the velocity sigmas.  

Decimate: Thins out velocity vectors on the map. The number indicates how many 1 in x vectors will be shown; so if it's 10, then 1 in 10 vectors will be displayed on the map. 

Plot Velocities: Plots lines showing velocities; Relative Vel: Shows velocity relative to selected site

Arrow mm/yr: Set length velocity scale arrow.

Vel Scale: Sets the scale of velocities projection on the map.  Nominally, it relates mm/yr to kilometers on the map, i.e., 10 sets 1 mm/yr to be 10 km on the map.  

<span style="color:red">The color of the vectors shows vertical motions with reds being down, blues up, and black near zero.</span>.  
±U Rate: Set the up velocity when the color saturates

Base Map Choice sets the base map with topography, street map, or ArcGIS imagery.



## Using the Timeseries
Site Id + Source Dropdown + Add to list: Input a Site ID with the source you want to add to the Site List. 

Live Update: Check to download the latest time-series data for selected sites, even if a previous version of the data already exists in the Data/ folder. 

Site List + Plot!: Select the Sites to be graphed in the Site List Box, and click "Plot!" to graph them. Selecting multiple sites will overlay their data on top of each other. 

Show Breaks Data: This button will show table(s) with all the breaks data available for the site(s), such as time, cause, type, and offsets. 

Remove Site: Removes the selected sites from the graph. 

Clear List + Close Graph: Self-explanatory.

Start/End: Set start and end dates for the data in the graph. Will only accept dates in the yyyy-mm-dd or yyyy-m-d format. 

SigLim: The limits on the standard deviations of the points of the time-series plot expressed in (north, east, up). Sigma must be less than the values of each component. 

Remove N σ: Only works if Detrend is also checked. Removes outlier data from the graphs. The number determines how strict the outlier determination is; all data outside the standard deviation multiplied by the Remove N σ number will be removed. The data is then detrended and run through the process again until the beginning and ending number of data points remain the same. The number of removed data points and the number of iterations will be given. 

Error Bars + Error Bar Outlines: Shows Error Bars and Error Bar outlines for the data. 

Error Bar Opacity + NErrBar: Changes the opacity of error bars and shows only every N'th errorbar. 

Shift Values + Update Shift: Shifts all the values of the selected sites up or down by the given mm. It is useful to align or separate two or more graph lines for better comparison. Click "Update Shift" to add the given shift value to the site(s). A dictionary displaying the offsets for each site is shown in the window  (if it is not shifted, it will not show up in the dictionary).  All sites can be selected to set the shift back to zero.

Copy/Clear Breaks Data: If the selected sites are also available in the NGF data, it will take the NGF break data and add it to the non-NGF version of the site, allowing for the "remove breaks" and "show breaks data" button to be available. Copying the NGF breaks can eb done for all sites in JPL/UNR time-series can be found in the availability section above. 

Backend Choice allows of type of time-series graphics.  Backend Activate <b>must</b> be used to activate the chosen backend.  
Errors activating the chosen BackEnd are displayed at the bottom of the time-series output window.  The ipympl Backend may need additional package installation.  This Backend allows the time-series plot to be Zoomed and Saved.  The Interactive Backend allows Zooming, Saving, and coordinates to be identified on the time-series plot.  When interactive is first selected, make sure to activate the Backend and check the error to see that it is activated without errors. 

On some systems, once the Backend is changed, the kernel needs to be restarted to change to a different backend.

ID points: Allows locations on the time-series plot to be identified only when the interactive backend is used.  The output includes the coordinates of the locations selected, and GLOBK site rename commands that can be used to add times of breaks in GLOBK/TSFIT solutions.  Multiple points can be selected with the middle mouse button/return ending selections, the right mouse button/delete removing points, and the left button/click selecting next point.  The graphics window needs to be active before the first point is selected. 

Detrended output example

| Site      |    Vn |   σVn |     Ve |   σVe |    Vu |   σVu |   WRMS N |   χn |   WRMS E |   χe |   WRMS U |   χu |   Num |
|:----------|------:|------:|-------:|------:|------:|------:|---------:|-----:|---------:|-----:|---------:|-----:|------:|
| P040-UNR  | -4.84 | 0.002 | -14.39 | 0.001 | -0.07 | 0.006 | 1.55     | 1.87 | 1.61     | 2.38 | 5.57     | 2.08 | 7146  |
| P040-JPL  | -4.86 | 0.002 | -14.41 | 0.001 | -0.01 | 0.005 | 1.66     | 2.24 | 1.54     | 2.54 | 5.41     | 2.26 | 7031  |
| P040-NGF | -4.96 | 0.004 | -14.34 | 0.003 | -0.32|  0.015 | 0.86     | 0.44 | 0.85     | 0.53 | 6.17     | 0.86 | 7262  |



In [ ]:
display(map_window) # displays map and map settings

In [ ]:
display(timeseries_window) # displays settings for graph

In [ ]:
display(timeseries_output_window) # displays graph, breaks data, and shift tracker (tied to "Update Shift")

In [ ]:
# import requests (Check login on Earthscope; maybe needed for EarthScope_sdk version 0.x)
NGF_test = False
#NGF_test = True
if ( NGF_test and esver[0]== '0' ) :
    url = "https://gage-data.earthscope.org/archive/gnss/products/velocity/cwu.fanet_igs14.vel"
    filename = "cwu.fanet_igs14.vel"
    print("Getting ",url)
    print("Link for login may appear below")
    get_es_file(url)
